In [20]:
import subprocess
import librosa
import numpy as np
import json
from pytube import YouTube
import os
import time
import re
import urllib.parse as urlparse

# Sample JSON timestamps for the first recording
timestamps_json = '''
[{"t":0,"mix":0},{"t":3.961,"mix":1},{"t":5.501,"mix":2},{"t":9.132,"mix":3},{"t":10.553,"mix":4},{"t":11.931,"mix":5},{"t":13.508,"mix":6},{"t":14.933,"mix":7},{"t":16.562,"mix":8},{"t":19.334,"mix":9},{"t":20.636,"mix":10},{"t":22.54,"mix":11},{"t":26.211,"mix":12},{"t":33.775,"mix":13},{"t":35.509,"mix":14},{"t":37.126,"mix":15},{"t":39.228,"mix":16},{"t":40.675,"mix":17},{"t":42.004,"mix":18},{"t":43.441,"mix":19},{"t":45.04,"mix":20},{"t":47.708,"mix":21},{"t":50.875,"mix":22},{"t":55.543,"mix":23},{"t":56.569,"mix":24},{"t":57.889,"mix":25},{"t":59.179,"mix":26},{"t":60.677,"mix":27},{"t":61.635,"mix":28},{"t":62.818,"mix":29},{"t":64.299,"mix":30},{"t":65.597,"mix":31},{"t":66.814,"mix":32},{"t":68.087,"mix":33},{"t":69.56,"mix":34},{"t":72.187,"mix":35},{"t":74.129,"mix":36},{"t":75.774,"mix":37},{"t":77.289,"mix":38},{"t":78.813,"mix":39},{"t":80.048,"mix":40},{"t":81.657,"mix":41},{"t":82.843,"mix":42},{"t":84.359,"mix":43},{"t":85.619,"mix":44},{"t":86.873,"mix":45},{"t":88.075,"mix":46},{"t":89.466,"mix":47},{"t":90.886,"mix":48},{"t":92.014,"mix":49},{"t":93.305,"mix":50},{"t":94.502,"mix":51},{"t":95.548,"mix":52},{"t":97.414,"mix":53},{"t":99.185,"mix":54},{"t":100.572,"mix":55},{"t":101.825,"mix":56},{"t":103.287,"mix":57},{"t":104.792,"mix":58},{"t":105.966,"mix":59},{"t":107.343,"mix":60},{"t":108.581,"mix":61},{"t":109.749,"mix":62},{"t":111.091,"mix":63},{"t":112.855,"mix":64},{"t":114.438,"mix":65},{"t":115.779,"mix":66},{"t":117.178,"mix":67},{"t":118.55,"mix":68},{"t":119.825,"mix":69},{"t":121.1,"mix":70},{"t":122.438,"mix":71},{"t":123.715,"mix":72},{"t":125.01,"mix":73},{"t":126.131,"mix":74},{"t":127.337,"mix":75},{"t":128.503,"mix":76},{"t":129.711,"mix":77},{"t":130.92,"mix":78},{"t":132.106,"mix":79},{"t":133.307,"mix":80},{"t":134.8,"mix":81},{"t":136.042,"mix":82},{"t":137.136,"mix":83},{"t":138.645,"mix":84},{"t":140.024,"mix":85},{"t":141.62,"mix":86},{"t":142.747,"mix":87},{"t":144.467,"mix":88},{"t":145.872,"mix":89},{"t":147.176,"mix":90},{"t":148.352,"mix":91},{"t":150.22,"mix":92},{"t":152.555,"mix":93},{"t":154.336,"mix":94},{"t":156.094,"mix":95},{"t":157.761,"mix":96},{"t":159.128,"mix":97},{"t":160.584,"mix":98},{"t":161.849,"mix":99},{"t":163.56,"mix":100},{"t":164.752,"mix":101},{"t":165.882,"mix":102},{"t":167.116,"mix":103},{"t":168.47,"mix":104},{"t":170.017,"mix":105},{"t":171.269,"mix":106},{"t":172.828,"mix":107},{"t":173.858,"mix":108},{"t":174.732,"mix":109},{"t":175.869,"mix":110},{"t":176.791,"mix":111},{"t":177.696,"mix":112},{"t":178.556,"mix":113},{"t":179.714,"mix":114},{"t":180.551,"mix":115},{"t":182.183,"mix":116},{"t":183.624,"mix":117},{"t":184.771,"mix":118},{"t":185.873,"mix":119},{"t":186.845,"mix":120},{"t":187.656,"mix":121},{"t":188.519,"mix":122},{"t":189.324,"mix":123},{"t":190.176,"mix":124},{"t":192.648,"mix":125},{"t":207.84,"mix":126},{"t":212.529,"mix":127},{"t":215.43,"mix":128},{"t":219.087,"mix":129},{"t":222.678,"mix":130},{"t":225.845,"mix":131},{"t":228.814,"mix":132},{"t":232.174,"mix":133},{"t":234.865,"mix":134},{"t":238.276,"mix":135},{"t":242.636,"mix":136},{"t":245.929,"mix":137},{"t":249.879,"mix":138},{"t":253.67,"mix":139},{"t":256.208,"mix":140},{"t":259.267,"mix":141},{"t":262.289,"mix":142},{"t":264.839,"mix":143},{"t":268.135,"mix":144},{"t":271.617,"mix":145},{"t":274.502,"mix":146},{"t":276.938,"mix":147},{"t":280.116,"mix":148},{"t":283.468,"mix":149},{"t":287.451,"mix":150},{"t":290.859,"mix":151},{"t":293.722,"mix":152},{"t":296.599,"mix":153},{"t":299.489,"mix":154},{"t":302.052,"mix":155},{"t":304.812,"mix":156},{"t":308.203,"mix":157},{"t":311.315,"mix":158},{"t":313.608,"mix":159},{"t":316.614,"mix":160},{"t":319.191,"mix":161},{"t":324.169,"mix":162},{"t":327.513,"mix":163},{"t":329.817,"mix":164},{"t":332.986,"mix":165},{"t":335.801,"mix":166},{"t":338.386,"mix":167},{"t":341.533,"mix":168},{"t":345.206,"mix":169},{"t":347.777,"mix":170},{"t":350.517,"mix":171},{"t":353.601,"mix":172},{"t":356.751,"mix":173},{"t":359.428,"mix":174},{"t":362.201,"mix":175},{"t":365.586,"mix":176},{"t":369.064,"mix":177},{"t":371.673,"mix":178},{"t":375.186,"mix":179},{"t":378.318,"mix":180},{"t":380.967,"mix":181},{"t":384.481,"mix":182},{"t":387.294,"mix":183},{"t":389.772,"mix":184},{"t":392.237,"mix":185},{"t":394.949,"mix":186},{"t":397.516,"mix":187},{"t":399.857,"mix":188},{"t":402.399,"mix":189},{"t":405.313,"mix":190},{"t":408.225,"mix":191},{"t":410.022,"mix":192},{"t":411.806,"mix":193},{"t":413.489,"mix":194},{"t":415.304,"mix":195},{"t":417.932,"mix":196},{"t":421.128,"mix":197},{"t":423.651,"mix":198},{"t":426.363,"mix":199},{"t":428.912,"mix":200},{"t":431.526,"mix":201},{"t":433.92,"mix":202},{"t":436.751,"mix":203},{"t":439.551,"mix":204},{"t":441.833,"mix":205},{"t":447.115,"mix":206},{"t":450.166,"mix":207},{"t":453.105,"mix":208},{"t":455.899,"mix":209},{"t":458.439,"mix":210},{"t":461.559,"mix":211},{"t":464.359,"mix":212},{"t":467.201,"mix":213},{"t":469.795,"mix":214},{"t":472.371,"mix":215},{"t":475.558,"mix":216},{"t":478.951,"mix":217},{"t":481.577,"mix":218},{"t":484.458,"mix":219},{"t":487.272,"mix":220},{"t":493.279,"mix":221},{"t":502.015,"mix":222},{"t":512.151,"mix":223},{"t":520.003,"mix":224},{"t":523.357,"mix":225},{"t":524.8,"mix":226},{"t":525.869,"mix":227},{"t":527.12,"mix":228},{"t":528.645,"mix":229},{"t":532.147,"mix":230},{"t":536.021,"mix":231},{"t":540.924,"mix":232},{"t":543.321,"mix":233},{"t":545.362,"mix":234},{"t":547.112,"mix":235},{"t":548.984,"mix":236},{"t":550.929,"mix":237},{"t":552.848,"mix":238},{"t":554.556,"mix":239},{"t":556.577,"mix":240},{"t":558.469,"mix":241},{"t":560.39,"mix":242},{"t":562.159,"mix":243},{"t":564.039,"mix":244},{"t":565.885,"mix":245},{"t":567.772,"mix":246},{"t":569.652,"mix":247},{"t":571.708,"mix":248},{"t":573.273,"mix":249},{"t":575.268,"mix":250},{"t":578.979,"mix":251},{"t":582.302,"mix":252},{"t":585.108,"mix":253},{"t":586.452,"mix":254},{"t":588.688,"mix":255},{"t":591.536,"mix":256},{"t":595.883,"mix":257},{"t":597.668,"mix":258},{"t":599.196,"mix":259},{"t":600.996,"mix":260},{"t":602.443,"mix":261},{"t":604.014,"mix":262},{"t":605.601,"mix":263},{"t":607.189,"mix":264},{"t":608.797,"mix":265},{"t":610.205,"mix":266},{"t":611.739,"mix":267},{"t":613.221,"mix":268},{"t":615.389,"mix":269},{"t":616.89,"mix":270},{"t":618.274,"mix":271},{"t":619.676,"mix":272},{"t":621.992,"mix":273},{"t":623.772,"mix":274},{"t":625.416,"mix":275},{"t":626.911,"mix":276},{"t":628.369,"mix":277},{"t":630.167,"mix":278},{"t":631.844,"mix":279},{"t":633.36,"mix":280},{"t":635.093,"mix":281},{"t":637.047,"mix":282},{"t":638.578,"mix":283},{"t":639.917,"mix":284},{"t":641.534,"mix":285},{"t":643.318,"mix":286},{"t":644.82,"mix":287},{"t":647.766,"mix":288},{"t":655.755,"mix":289},{"t":658.826,"mix":290},{"t":660.978,"mix":291},{"t":663.212,"mix":292},{"t":665.113,"mix":293},{"t":667.512,"mix":294},{"t":669.506,"mix":295},{"t":671.852,"mix":296},{"t":673.958,"mix":297},{"t":676.369,"mix":298},{"t":678.396,"mix":299},{"t":680.488,"mix":300},{"t":682.932,"mix":301},{"t":686.262,"mix":302},{"t":688.753,"mix":303},{"t":693.065,"mix":304},{"t":696.612,"mix":305},{"t":698.421,"mix":306},{"t":704.737,"mix":307},{"t":706.087,"mix":308},{"t":708.233,"mix":309},{"t":710.458,"mix":310},{"t":712.53,"mix":311},{"t":714.332,"mix":312},{"t":716.401,"mix":313},{"t":718.148,"mix":314},{"t":720.134,"mix":315},{"t":721.997,"mix":316},{"t":724.126,"mix":317},{"t":726.018,"mix":318},{"t":727.92,"mix":319},{"t":730.36,"mix":320},{"t":732.303,"mix":321},{"t":733.93,"mix":322},{"t":735.354,"mix":323},{"t":736.963,"mix":324},{"t":739.24,"mix":325},{"t":741.373,"mix":326},{"t":743.197,"mix":327},{"t":745.086,"mix":328},{"t":746.762,"mix":329},{"t":748.192,"mix":330},{"t":749.667,"mix":331},{"t":751.089,"mix":332},{"t":753.379,"mix":333},{"t":754.884,"mix":334},{"t":755.903,"mix":335},{"t":756.854,"mix":336},{"t":759.002,"mix":337},{"t":761.424,"mix":338},{"t":763.194,"mix":339},{"t":765.241,"mix":340},{"t":766.733,"mix":341},{"t":771.52,"mix":342},{"t":778.987,"mix":343},{"t":780.741,"mix":344},{"t":782.304,"mix":345},{"t":783.815,"mix":346},{"t":785.378,"mix":347},{"t":786.926,"mix":348},{"t":788.516,"mix":349},{"t":789.94,"mix":350},{"t":791.471,"mix":351},{"t":793.208,"mix":352},{"t":794.698,"mix":353},{"t":796.156,"mix":354},{"t":797.689,"mix":355},{"t":799.078,"mix":356},{"t":800.415,"mix":357},{"t":802.159,"mix":358},{"t":803.532,"mix":359},{"t":804.847,"mix":360},{"t":806.291,"mix":361},{"t":807.912,"mix":362},{"t":809.354,"mix":363},{"t":810.775,"mix":364},{"t":813.279,"mix":365},{"t":814.295,"mix":366},{"t":815.389,"mix":367},{"t":816.46,"mix":368},{"t":817.504,"mix":369},{"t":818.515,"mix":370},{"t":819.537,"mix":371},{"t":820.515,"mix":372},{"t":821.521,"mix":373},{"t":822.573,"mix":374},{"t":823.604,"mix":375},{"t":824.581,"mix":376},{"t":825.655,"mix":377},{"t":826.707,"mix":378},{"t":827.681,"mix":379},{"t":828.945,"mix":380},{"t":830.017,"mix":381},{"t":830.875,"mix":382},{"t":831.698,"mix":383},{"t":832.534,"mix":384},{"t":833.293,"mix":385},{"t":834.039,"mix":386},{"t":834.973,"mix":387},{"t":837.427,"mix":388},{"t":839.314,"mix":389},{"t":840.753,"mix":390},{"t":842.266,"mix":391},{"t":844.046,"mix":392},{"t":845.771,"mix":393},{"t":847.327,"mix":394},{"t":848.831,"mix":395},{"t":850.898,"mix":396},{"t":852.372,"mix":397},{"t":853.361,"mix":398},{"t":854.412,"mix":399},{"t":855.475,"mix":400},{"t":856.377,"mix":401},{"t":857.277,"mix":402},{"t":858.174,"mix":403},{"t":859.204,"mix":404},{"t":860.601,"mix":405},{"t":861.505,"mix":406},{"t":881.429,"mix":407},{"t":886.37,"mix":408},{"t":888.919,"mix":409},{"t":891.76,"mix":410},{"t":894.452,"mix":411},{"t":900.634,"mix":412},{"t":906.374,"mix":413},{"t":912.648,"mix":414},{"t":918.246,"mix":415},{"t":925.337,"mix":416},{"t":930.837,"mix":417},{"t":939.181,"mix":418},{"t":943.328,"mix":419},{"t":947.525,"mix":420},{"t":950.231,"mix":421},{"t":952.197,"mix":422},{"t":955.022,"mix":423},{"t":956.822,"mix":424},{"t":957.93,"mix":425},{"t":959.075,"mix":426},{"t":960.34,"mix":427},{"t":962.554,"mix":428},{"t":964.386,"mix":429},{"t":967.483,"mix":430},{"t":984.842,"mix":431},{"t":986.694,"mix":432},{"t":988.022,"mix":433},{"t":989.232,"mix":434},{"t":990.291,"mix":435},{"t":991.57,"mix":436},{"t":992.838,"mix":437},{"t":994.24,"mix":438},{"t":995.586,"mix":439},{"t":997.113,"mix":440},{"t":998.294,"mix":441},{"t":1000.091,"mix":442},{"t":1002.333,"mix":443},{"t":1003.693,"mix":444},{"t":1005.456,"mix":445},{"t":1007.248,"mix":446},{"t":1009.037,"mix":447},{"t":1010.382,"mix":448},{"t":1012.038,"mix":449},{"t":1013.59,"mix":450},{"t":1015.197,"mix":451},{"t":1016.705,"mix":452},{"t":1018.339,"mix":453},{"t":1019.837,"mix":454},{"t":1021.401,"mix":455},{"t":1022.859,"mix":456},{"t":1024.685,"mix":457},{"t":1026.179,"mix":458},{"t":1027.712,"mix":459},{"t":1029.215,"mix":460},{"t":1030.886,"mix":461},{"t":1032.312,"mix":462},{"t":1033.848,"mix":463},{"t":1035.28,"mix":464},{"t":1036.683,"mix":465},{"t":1038.117,"mix":466},{"t":1039.531,"mix":467},{"t":1040.953,"mix":468},{"t":1042.601,"mix":469},{"t":1044.487,"mix":470},{"t":1046.3,"mix":471},{"t":1048.589,"mix":472},{"t":1050.508,"mix":473},{"t":1052.113,"mix":474},{"t":1053.486,"mix":475},{"t":1055.003,"mix":476},{"t":1056.528,"mix":477},{"t":1058.335,"mix":478},{"t":1059.792,"mix":479},{"t":1061.197,"mix":480},{"t":1062.65,"mix":481},{"t":1064.112,"mix":482},{"t":1065.514,"mix":483},{"t":1066.964,"mix":484},{"t":1068.271,"mix":485},{"t":1069.661,"mix":486},{"t":1071.134,"mix":487},{"t":1072.484,"mix":488},{"t":1073.794,"mix":489},{"t":1075.27,"mix":490},{"t":1076.695,"mix":491},{"t":1078.277,"mix":492},{"t":1079.788,"mix":493},{"t":1081.368,"mix":494},{"t":1082.853,"mix":495},{"t":1084.599,"mix":496},{"t":1086.053,"mix":497},{"t":1087.362,"mix":498},{"t":1088.689,"mix":499},{"t":1090.067,"mix":500},{"t":1091.496,"mix":501},{"t":1092.875,"mix":502},{"t":1094.347,"mix":503},{"t":1095.926,"mix":504},{"t":1097.377,"mix":505},{"t":1098.785,"mix":506},{"t":1100.24,"mix":507},{"t":1101.715,"mix":508},{"t":1103.089,"mix":509},{"t":1104.659,"mix":510},{"t":1106.08,"mix":511},{"t":1107.662,"mix":512},{"t":1109.16,"mix":513},{"t":1110.644,"mix":514},{"t":1111.94,"mix":515},{"t":1113.245,"mix":516},{"t":1114.565,"mix":517},{"t":1115.94,"mix":518},{"t":1117.233,"mix":519},{"t":1118.537,"mix":520},{"t":1119.825,"mix":521},{"t":1121.166,"mix":522},{"t":1122.823,"mix":523},{"t":1124.288,"mix":524},{"t":1125.856,"mix":525},{"t":1127.452,"mix":526},{"t":1129.089,"mix":527},{"t":1130.649,"mix":528},{"t":1132.609,"mix":529},{"t":1134.452,"mix":530},{"t":1136.237,"mix":531},{"t":1138.598,"mix":532},{"t":1141.031,"mix":533},{"t":1142.844,"mix":534},{"t":1145.088,"mix":535},{"t":1147.462,"mix":536},{"t":1148.667,"mix":537},{"t":1150.021,"mix":538},{"t":1151.233,"mix":539},{"t":1152.805,"mix":540},{"t":1154.072,"mix":541},{"t":1155.443,"mix":542},{"t":1157.321,"mix":543},{"t":1158.928,"mix":544},{"t":1160.178,"mix":545},{"t":1161.483,"mix":546},{"t":1162.857,"mix":547},{"t":1164.327,"mix":548},{"t":1165.664,"mix":549},{"t":1167.05,"mix":550},{"t":1168.448,"mix":551},{"t":1170.241,"mix":552},{"t":1171.746,"mix":553},{"t":1173.227,"mix":554},{"t":1174.702,"mix":555},{"t":1176.113,"mix":556},{"t":1177.737,"mix":557},{"t":1179.361,"mix":558},{"t":1180.5,"mix":559},{"t":1182.018,"mix":560},{"t":1183.377,"mix":561},{"t":1184.825,"mix":562},{"t":1186.953,"mix":563},{"t":1212.271,"mix":564},{"t":1218.804,"mix":565},{"t":1223.844,"mix":566},{"t":1229.537,"mix":567},{"t":1235.1,"mix":568},{"t":1240.322,"mix":569},{"t":1245.881,"mix":570},{"t":1251.839,"mix":571},{"t":1257.15,"mix":572},{"t":1261.414,"mix":573},{"t":1265.924,"mix":574},{"t":1270.046,"mix":575},{"t":1275.083,"mix":576},{"t":1280.08,"mix":577},{"t":1284.803,"mix":578},{"t":1289.494,"mix":579},{"t":1294.164,"mix":580},{"t":1298.729,"mix":581},{"t":1303.315,"mix":582},{"t":1307.915,"mix":583},{"t":1313.079,"mix":584},{"t":1317.09,"mix":585},{"t":1321.031,"mix":586},{"t":1324.345,"mix":587},{"t":1328.052,"mix":588},{"t":1330.859,"mix":589},{"t":1333.558,"mix":590},{"t":1336.759,"mix":591},{"t":1339.566,"mix":592},{"t":1342.141,"mix":593},{"t":1345.121,"mix":594},{"t":1351.182,"mix":595},{"t":1355.683,"mix":596},{"t":1360.271,"mix":597},{"t":1364.877,"mix":598},{"t":1369.578,"mix":599},{"t":1373.978,"mix":600},{"t":1378.724,"mix":601},{"t":1383.332,"mix":602},{"t":1388.208,"mix":603},{"t":1391.604,"mix":604},{"t":1396.407,"mix":605},{"t":1399.546,"mix":606},{"t":1404.421,"mix":607},{"t":1408.805,"mix":608},{"t":1412.856,"mix":609},{"t":1416.608,"mix":610},{"t":1421.359,"mix":611},{"t":1425.484,"mix":612},{"t":1429.712,"mix":613},{"t":1433.51,"mix":614},{"t":1437.995,"mix":615},{"t":1442.454,"mix":616},{"t":1446.908,"mix":617},{"t":1453.201,"mix":618},{"t":1459.747,"mix":619},{"t":1463.401,"mix":620},{"t":1467.74,"mix":621},{"t":1471.621,"mix":622},{"t":1476.314,"mix":623},{"t":1480.077,"mix":624},{"t":1484.644,"mix":625},{"t":1488.74,"mix":626},{"t":1494.535,"mix":627},{"t":1500.499,"mix":628},{"t":1505.332,"mix":629},{"t":1511.265,"mix":630},{"t":1516.838,"mix":631},{"t":1521.653,"mix":632},{"t":1526.794,"mix":633},{"t":1543.573,"mix":634},{"t":1546.085,"mix":635},{"t":1548.655,"mix":636},{"t":1551.273,"mix":637},{"t":1553.829,"mix":638},{"t":1555.75,"mix":639},{"t":1558.589,"mix":640},{"t":1560.908,"mix":641},{"t":1563.389,"mix":642},{"t":1565.676,"mix":643},{"t":1567.4,"mix":644},{"t":1569.151,"mix":645},{"t":1571.048,"mix":646},{"t":1572.779,"mix":647},{"t":1575.07,"mix":648},{"t":1577.249,"mix":649},{"t":1579.387,"mix":650},{"t":1581.034,"mix":651},{"t":1583.799,"mix":652},{"t":1586.938,"mix":653},{"t":1588.226,"mix":654},{"t":1590.771,"mix":655},{"t":1593.917,"mix":656},{"t":1596.486,"mix":657},{"t":1599.482,"mix":658},{"t":1601.762,"mix":659},{"t":1604.318,"mix":660},{"t":1606.469,"mix":661},{"t":1608.562,"mix":662},{"t":1611.215,"mix":663},{"t":1615.015,"mix":664},{"t":1618.073,"mix":665},{"t":1620.867,"mix":666},{"t":1623.183,"mix":667},{"t":1625.013,"mix":668},{"t":1627.122,"mix":669},{"t":1629.239,"mix":670},{"t":1631.126,"mix":671},{"t":1632.998,"mix":672},{"t":1635.004,"mix":673},{"t":1637.009,"mix":674},{"t":1638.985,"mix":675},{"t":1641.417,"mix":676},{"t":1643.845,"mix":677},{"t":1646.958,"mix":678},{"t":1648.969,"mix":679},{"t":1652.29,"mix":680},{"t":1654.827,"mix":681},{"t":1657.819,"mix":682},{"t":1660.496,"mix":683},{"t":1662.82,"mix":684},{"t":1665.851,"mix":685},{"t":1668.148,"mix":686},{"t":1670.303,"mix":687},{"t":1673.224,"mix":688},{"t":1676.209,"mix":689},{"t":1678.077,"mix":690},{"t":1680.009,"mix":691},{"t":1681.942,"mix":692},{"t":1683.701,"mix":693},{"t":1685.707,"mix":694},{"t":1687.66,"mix":695},{"t":1689.597,"mix":696},{"t":1691.593,"mix":697},{"t":1693.585,"mix":698},{"t":1695.393,"mix":699},{"t":1697.308,"mix":700},{"t":1699.108,"mix":701},{"t":1700.925,"mix":702},{"t":1702.736,"mix":703},{"t":1704.718,"mix":704},{"t":1706.643,"mix":705},{"t":1708.367,"mix":706},{"t":1710.433,"mix":707},{"t":1712.314,"mix":708},{"t":1713.92,"mix":709},{"t":1716.668,"mix":710},{"t":1718.632,"mix":711},{"t":1720.472,"mix":712},{"t":1721.885,"mix":713},{"t":1723.349,"mix":714},{"t":1724.866,"mix":715},{"t":1726.501,"mix":716},{"t":1728.607,"mix":717},{"t":1730.386,"mix":718},{"t":1732.057,"mix":719},{"t":1735.454,"mix":720},{"t":1737.291,"mix":721},{"t":1739.066,"mix":722},{"t":1740.874,"mix":723},{"t":1742.534,"mix":724},{"t":1744.286,"mix":725},{"t":1746.151,"mix":726},{"t":1748.088,"mix":727},{"t":1749.823,"mix":728},{"t":1751.751,"mix":729},{"t":1753.437,"mix":730},{"t":1755.348,"mix":731},{"t":1757.125,"mix":732},{"t":1758.95,"mix":733},{"t":1760.677,"mix":734},{"t":1762.801,"mix":735},{"t":1764.571,"mix":736},{"t":1768.313,"mix":737},{"t":1770.887,"mix":738},{"t":1773.973,"mix":739},{"t":1777.863,"mix":740},{"t":1780.935,"mix":741},{"t":1785.25,"mix":742},{"t":1787.373,"mix":743},{"t":1789.942,"mix":744},{"t":1792.083,"mix":745},{"t":1795.03,"mix":746},{"t":1796.491,"mix":747},{"t":1798.624,"mix":748},{"t":1800.959,"mix":749},{"t":1803.531,"mix":750},{"t":1805.893,"mix":751},{"t":1809.4,"mix":752},{"t":1815.157,"mix":753},{"t":1836.18,"mix":754},{"t":1837.989,"mix":755},{"t":1839.369,"mix":756},{"t":1840.546,"mix":757},{"t":1841.575,"mix":758},{"t":1842.934,"mix":759},{"t":1844.001,"mix":760},{"t":1845.244,"mix":761},{"t":1846.484,"mix":762},{"t":1847.834,"mix":763},{"t":1849.122,"mix":764},{"t":1850.202,"mix":765},{"t":1851.781,"mix":766},{"t":1852.845,"mix":767},{"t":1853.926,"mix":768},{"t":1855.266,"mix":769},{"t":1856.735,"mix":770},{"t":1858.455,"mix":771},{"t":1859.219,"mix":772},{"t":1860.788,"mix":773},{"t":1861.825,"mix":774},{"t":1862.965,"mix":775},{"t":1864.053,"mix":776},{"t":1865.19,"mix":777},{"t":1866.531,"mix":778},{"t":1867.56,"mix":779},{"t":1868.885,"mix":780},{"t":1870.839,"mix":781},{"t":1871.977,"mix":782},{"t":1873.465,"mix":783},{"t":1875,"mix":784},{"t":1875.68,"mix":785},{"t":1876.934,"mix":786},{"t":1878.468,"mix":787},{"t":1879.66,"mix":788},{"t":1880.683,"mix":789},{"t":1882.207,"mix":790},{"t":1883.171,"mix":791},{"t":1884.446,"mix":792},{"t":1885.748,"mix":793},{"t":1886.795,"mix":794},{"t":1888.021,"mix":795},{"t":1889.245,"mix":796},{"t":1890.857,"mix":797},{"t":1891.928,"mix":798},{"t":1892.919,"mix":799},{"t":1894.159,"mix":800},{"t":1895.605,"mix":801},{"t":1896.869,"mix":802},{"t":1898.19,"mix":803},{"t":1899.959,"mix":804},{"t":1900.886,"mix":805},{"t":1901.811,"mix":806},{"t":1902.667,"mix":807},{"t":1903.613,"mix":808},{"t":1904.8,"mix":809},{"t":1905.764,"mix":810},{"t":1906.758,"mix":811},{"t":1908.053,"mix":812},{"t":1909.281,"mix":813},{"t":1910.448,"mix":814},{"t":1911.525,"mix":815},{"t":1912.723,"mix":816},{"t":1913.854,"mix":817},{"t":1915.829,"mix":818},{"t":1917.045,"mix":819},{"t":1918.711,"mix":820},{"t":1919.892,"mix":821},{"t":1921.044,"mix":822},{"t":1922.136,"mix":823},{"t":1923.229,"mix":824},{"t":1924.224,"mix":825},{"t":1925.293,"mix":826},{"t":1926.401,"mix":827},{"t":1927.435,"mix":828},{"t":1928.479,"mix":829},{"t":1929.607,"mix":830},{"t":1930.665,"mix":831},{"t":1931.69,"mix":832},{"t":1932.735,"mix":833},{"t":1933.791,"mix":834},{"t":1934.888,"mix":835},{"t":1935.944,"mix":836},{"t":1936.982,"mix":837},{"t":1940.538,"mix":838},{"t":1942.177,"mix":839},{"t":1943.619,"mix":840},{"t":1945.014,"mix":841},{"t":1946.867,"mix":842},{"t":1948.044,"mix":843},{"t":1949.431,"mix":844},{"t":1950.754,"mix":845},{"t":1952.804,"mix":846},{"t":1954.275,"mix":847},{"t":1955.42,"mix":848},{"t":1956.777,"mix":849},{"t":1958.121,"mix":850},{"t":1959.309,"mix":851},{"t":1960.43,"mix":852},{"t":1961.953,"mix":853},{"t":1963.853,"mix":854},{"t":1965.356,"mix":855},{"t":1966.48,"mix":856},{"t":1967.752,"mix":857},{"t":1969.118,"mix":858},{"t":1970.323,"mix":859},{"t":1971.47,"mix":860},{"t":1972.688,"mix":861},{"t":1974.044,"mix":862},{"t":1975.202,"mix":863},{"t":1976.454,"mix":864},{"t":1977.595,"mix":865},{"t":1978.671,"mix":866},{"t":1979.774,"mix":867},{"t":1980.843,"mix":868},{"t":1982.768,"mix":869},{"t":1984.437,"mix":870},{"t":1985.673,"mix":871},{"t":1986.951,"mix":872},{"t":1988.472,"mix":873},{"t":1989.92,"mix":874},{"t":1991.089,"mix":875},{"t":1992.749,"mix":876},{"t":1994.199,"mix":877},{"t":1995.707,"mix":878},{"t":1996.903,"mix":879},{"t":1998.072,"mix":880},{"t":1999.232,"mix":881},{"t":2000.295,"mix":882},{"t":2001.436,"mix":883},{"t":2002.558,"mix":884},{"t":2003.732,"mix":885},{"t":2005.211,"mix":886},{"t":2006.876,"mix":887},{"t":2008.041,"mix":888},{"t":2009.048,"mix":889},{"t":2009.96,"mix":890},{"t":2011.014,"mix":891},{"t":2011.93,"mix":892},{"t":2012.895,"mix":893},{"t":2013.792,"mix":894},{"t":2014.82,"mix":895},{"t":2015.753,"mix":896},{"t":2016.746,"mix":897},{"t":2017.775,"mix":898},{"t":2019.09,"mix":899},{"t":2020.105,"mix":900},{"t":2021.035,"mix":901},{"t":2022.071,"mix":902},{"t":2023.095,"mix":903},{"t":2024.182,"mix":904},{"t":2025.178,"mix":905},{"t":2026.174,"mix":906},{"t":2027.13,"mix":907},{"t":2028.098,"mix":908},{"t":2029.07,"mix":909},{"t":2029.988,"mix":910},{"t":2030.952,"mix":911},{"t":2031.892,"mix":912},{"t":2032.891,"mix":913},{"t":2033.844,"mix":914},{"t":2034.924,"mix":915},{"t":2035.986,"mix":916},{"t":2037.936,"mix":917},{"t":2039.359,"mix":918},{"t":2040.612,"mix":919},{"t":2041.584,"mix":920},{"t":2042.629,"mix":921},{"t":2043.64,"mix":922},{"t":2044.637,"mix":923},{"t":2045.661,"mix":924},{"t":2047.086,"mix":925},{"t":2048.041,"mix":926},{"t":2049.331,"mix":927},{"t":2050.563,"mix":928},{"t":2051.774,"mix":929},{"t":2052.876,"mix":930},{"t":2054.433,"mix":931},{"t":2055.386,"mix":932},{"t":2056.37,"mix":933},{"t":2057.309,"mix":934},{"t":2058.269,"mix":935},{"t":2059.29,"mix":936},{"t":2060.285,"mix":937},{"t":2061.234,"mix":938},{"t":2062.408,"mix":939},{"t":2063.422,"mix":940},{"t":2064.376,"mix":941},{"t":2065.441,"mix":942},{"t":2066.376,"mix":943},{"t":2067.357,"mix":944},{"t":2068.351,"mix":945},{"t":2069.396,"mix":946},{"t":2071.519,"mix":947},{"t":2073.062,"mix":948},{"t":2074.278,"mix":949},{"t":2075.449,"mix":950},{"t":2076.788,"mix":951},{"t":2078.146,"mix":952},{"t":2079.214,"mix":953},{"t":2080.239,"mix":954},{"t":2082.08,"mix":955},{"t":2083.419,"mix":956},{"t":2084.302,"mix":957},{"t":2085.236,"mix":958},{"t":2086.221,"mix":959},{"t":2087.291,"mix":960},{"t":2088.217,"mix":961},{"t":2089.09,"mix":962},{"t":2089.992,"mix":963},{"t":2090.756,"mix":964},{"t":2092.074,"mix":965},{"t":2093.537,"mix":966},{"t":2094.344,"mix":967},{"t":2095.255,"mix":968},{"t":2096.804,"mix":969},{"t":2097.835,"mix":970},{"t":2099.047,"mix":971},{"t":2100.316,"mix":972},{"t":2101.817,"mix":973},{"t":2102.976,"mix":974},{"t":2105.093,"mix":975},{"t":2105.777,"mix":976},{"t":2106.624,"mix":977},{"t":2107.442,"mix":978},{"t":2108.289,"mix":979},{"t":2109.232,"mix":980},{"t":2111.317,"mix":981},{"t":2167.962,"mix":982},{"t":2171.939,"mix":983},{"t":2177.795,"mix":984},{"t":2186.66,"mix":985},{"t":2192.894,"mix":986},{"t":2198.142,"mix":987},{"t":2203.767,"mix":988},{"t":2209.647,"mix":989},{"t":2216.403,"mix":990},{"t":2221.506,"mix":991},{"t":2230.668,"mix":992},{"t":2235.135,"mix":993},{"t":2238.758,"mix":994},{"t":2242.336,"mix":995},{"t":2245.642,"mix":996},{"t":2250.189,"mix":997},{"t":2253.504,"mix":998},{"t":2257.831,"mix":999},{"t":2261.553,"mix":1000},{"t":2279.963,"mix":1001},{"t":2283.441,"mix":1002},{"t":2287.158,"mix":1003},{"t":2293.55,"mix":1004},{"t":2298.36,"mix":1005},{"t":2303.913,"mix":1006},{"t":2309.564,"mix":1007},{"t":2315.467,"mix":1008},{"t":2321.103,"mix":1009},{"t":2326.093,"mix":1010},{"t":2332.884,"mix":1011},{"t":2337.466,"mix":1012},{"t":2341.048,"mix":1013},{"t":2345.001,"mix":1014},{"t":2349.66,"mix":1015},{"t":2352.835,"mix":1016},{"t":2356.681,"mix":1017},{"t":2361.078,"mix":1018},{"t":2366.251,"mix":1019},{"t":2370.494,"mix":1020},{"t":2376.1,"mix":1021},{"t":2379.694,"mix":1022},{"t":2383.415,"mix":1023},{"t":2389.132,"mix":1024},{"t":2393.021,"mix":1025},{"t":2397.152,"mix":1026},{"t":2400.968,"mix":1027},{"t":2405.787,"mix":1028},{"t":2409.001,"mix":1029},{"t":2412.838,"mix":1030},{"t":2415.932,"mix":1031},{"t":2433.484,"mix":1032},{"t":2439.405,"mix":1033},{"t":2444.2,"mix":1034},{"t":2449.033,"mix":1035},{"t":2453.77,"mix":1036},{"t":2459.367,"mix":1037},{"t":2465.022,"mix":1038},{"t":2470.312,"mix":1039},{"t":2480.261,"mix":1040},{"t":2487.472,"mix":1041},{"t":2494.215,"mix":1042},{"t":2501.382,"mix":1043},{"t":2508.687,"mix":1044},{"t":2516.237,"mix":1045},{"t":2523.164,"mix":1046},{"t":2528.968,"mix":1047},{"t":2534.313,"mix":1048},{"t":2538.254,"mix":1049},{"t":2541.49,"mix":1050},{"t":2544.881,"mix":1051},{"t":2549.236,"mix":1052},{"t":2553.758,"mix":1053},{"t":2557.739,"mix":1054},{"t":2561.933,"mix":1055},{"t":2565.363,"mix":1056},{"t":2569.713,"mix":1057},{"t":2574.272,"mix":1058},{"t":2577.25,"mix":1059},{"t":2581.438,"mix":1060},{"t":2585.248,"mix":1061},{"t":2589.008,"mix":1062},{"t":2593.223,"mix":1063},{"t":2615.775,"mix":1064},{"t":2621.444,"mix":1065},{"t":2626.34,"mix":1066},{"t":2631.951,"mix":1067},{"t":2638.753,"mix":1068},{"t":2643.465,"mix":1069},{"t":2648.996,"mix":1070},{"t":2654.018,"mix":1071},{"t":2658.194,"mix":1072},{"t":2661.823,"mix":1073},{"t":2665.13,"mix":1074},{"t":2668.803,"mix":1075},{"t":2672.898,"mix":1076},{"t":2676.515,"mix":1077},{"t":2679.596,"mix":1078},{"t":2682.134,"mix":1079},{"t":2689.457,"mix":1080},{"t":2700.51,"mix":1081},{"t":2708.828,"mix":1082},{"t":2716.08,"mix":1083},{"t":2724.005,"mix":1084},{"t":2733.953,"mix":1085},{"t":2742.58,"mix":1086},{"t":2750.424,"mix":1087},{"t":2756.573,"mix":1088},{"t":2764.173,"mix":1089},{"t":2774.342,"mix":1090},{"t":2780.693,"mix":1091},{"t":2790.2,"mix":1092},{"t":2797.307,"mix":1093},{"t":2802.745,"mix":1094},{"t":2809.504,"mix":1095},{"t":2813.483,"mix":1096},{"t":2817.288,"mix":1097},{"t":2821.24,"mix":1098},{"t":2825.587,"mix":1099},{"t":2835.897,"mix":1100},{"t":2852.911,"mix":1101},{"t":2854.103,"mix":1102},{"t":2855.103,"mix":1103},{"t":2856.419,"mix":1104},{"t":2857.491,"mix":1105},{"t":2858.493,"mix":1106},{"t":2859.668,"mix":1107},{"t":2861.322,"mix":1108},{"t":2862.465,"mix":1109},{"t":2864.027,"mix":1110},{"t":2865.145,"mix":1111},{"t":2866.706,"mix":1112},{"t":2867.88,"mix":1113},{"t":2868.875,"mix":1114},{"t":2869.943,"mix":1115},{"t":2870.936,"mix":1116},{"t":2872.213,"mix":1117},{"t":2873.712,"mix":1118},{"t":2874.787,"mix":1119},{"t":2876.211,"mix":1120},{"t":2877.555,"mix":1121},{"t":2879.488,"mix":1122},{"t":2881.118,"mix":1123},{"t":2882.66,"mix":1124},{"t":2883.707,"mix":1125},{"t":2885.303,"mix":1126},{"t":2886.683,"mix":1127},{"t":2888.231,"mix":1128},{"t":2889.594,"mix":1129},{"t":2891.158,"mix":1130},{"t":2893.408,"mix":1131},{"t":2895.284,"mix":1132},{"t":2896.768,"mix":1133},{"t":2898.246,"mix":1134},{"t":2899.9,"mix":1135},{"t":2901.322,"mix":1136},{"t":2902.735,"mix":1137},{"t":2904.535,"mix":1138},{"t":2905.982,"mix":1139},{"t":2907.601,"mix":1140},{"t":2909.042,"mix":1141},{"t":2910.857,"mix":1142},{"t":2911.931,"mix":1143},{"t":2912.959,"mix":1144},{"t":2913.873,"mix":1145},{"t":2914.829,"mix":1146},{"t":2915.806,"mix":1147},{"t":2916.884,"mix":1148},{"t":2917.832,"mix":1149},{"t":2918.869,"mix":1150},{"t":2919.77,"mix":1151},{"t":2920.814,"mix":1152},{"t":2921.838,"mix":1153},{"t":2923.791,"mix":1154},{"t":2925.629,"mix":1155},{"t":2927.253,"mix":1156},{"t":2928.888,"mix":1157},{"t":2930.332,"mix":1158},{"t":2931.497,"mix":1159},{"t":2933.065,"mix":1160},{"t":2934.603,"mix":1161},{"t":2936.574,"mix":1162},{"t":2937.988,"mix":1163},{"t":2939.357,"mix":1164},{"t":2940.884,"mix":1165},{"t":2942.016,"mix":1166},{"t":2942.977,"mix":1167},{"t":2943.976,"mix":1168},{"t":2945.235,"mix":1169},{"t":2946.378,"mix":1170},{"t":2947.427,"mix":1171},{"t":2948.564,"mix":1172},{"t":2949.666,"mix":1173},{"t":2950.685,"mix":1174},{"t":2951.774,"mix":1175},{"t":2953.116,"mix":1176},{"t":2954.039,"mix":1177},{"t":2955.166,"mix":1178},{"t":2955.833,"mix":1179},{"t":2956.479,"mix":1180},{"t":2957.093,"mix":1181},{"t":2957.685,"mix":1182},{"t":2958.257,"mix":1183},{"t":2958.825,"mix":1184},{"t":2959.474,"mix":1185},{"t":2960.453,"mix":1186},{"t":2962.254,"mix":1187},{"t":2963.363,"mix":1188},{"t":2964.375,"mix":1189},{"t":2967.422,"mix":1190},{"t":2969.075,"mix":1191},{"t":2970.622,"mix":1192},{"t":2971.819,"mix":1193},{"t":2973.307,"mix":1194},{"t":2974.867,"mix":1195},{"t":2976.576,"mix":1196},{"t":2978.079,"mix":1197},{"t":2979.683,"mix":1198},{"t":2982.325,"mix":1199},{"t":2984.189,"mix":1200},{"t":2985.633,"mix":1201},{"t":2987.09,"mix":1202},{"t":2988.485,"mix":1203},{"t":2990.216,"mix":1204},{"t":2991.563,"mix":1205},{"t":2993.103,"mix":1206},{"t":2994.991,"mix":1207},{"t":2996.693,"mix":1208},{"t":2998.33,"mix":1209},{"t":3000.052,"mix":1210},{"t":3001.864,"mix":1211},{"t":3003.427,"mix":1212},{"t":3005.309,"mix":1213},{"t":3006.881,"mix":1214},{"t":3008.533,"mix":1215},{"t":3010.371,"mix":1216},{"t":3012.523,"mix":1217},{"t":3014.099,"mix":1218},{"t":3015.949,"mix":1219},{"t":3017.529,"mix":1220},{"t":3019.007,"mix":1221},{"t":3020.361,"mix":1222},{"t":3021.812,"mix":1223},{"t":3023.147,"mix":1224},{"t":3025.105,"mix":1225},{"t":3027.949,"mix":1226},{"t":3030.055,"mix":1227},{"t":3031.356,"mix":1228},{"t":3032.849,"mix":1229},{"t":3034.113,"mix":1230},{"t":3035.647,"mix":1231},{"t":3037.047,"mix":1232},{"t":3038.734,"mix":1233},{"t":3040.572,"mix":1234},{"t":3041.831,"mix":1235},{"t":3043.705,"mix":1236},{"t":3044.518,"mix":1237},{"t":3045.503,"mix":1238},{"t":3046.405,"mix":1239},{"t":3047.417,"mix":1240},{"t":3048.352,"mix":1241},{"t":3049.544,"mix":1242},{"t":3050.498,"mix":1243},{"t":3051.644,"mix":1244},{"t":3052.744,"mix":1245},{"t":3053.789,"mix":1246},{"t":3055.739,"mix":1247},{"t":3057.791,"mix":1248},{"t":3059.487,"mix":1249},{"t":3061.223,"mix":1250},{"t":3062.791,"mix":1251},{"t":3064.152,"mix":1252},{"t":3065.265,"mix":1253},{"t":3066.692,"mix":1254},{"t":3068.019,"mix":1255},{"t":3069.343,"mix":1256},{"t":3070.665,"mix":1257},{"t":3071.888,"mix":1258},{"t":3073.273,"mix":1259},{"t":3077.998,"mix":1260},{"t":3078.899,"mix":1261},{"t":3079.88,"mix":1262},{"t":3080.907,"mix":1263},{"t":3081.975,"mix":1264},{"t":3083.002,"mix":1265},{"t":3083.942,"mix":1266},{"t":3085.05,"mix":1267},{"t":3086.127,"mix":1268},{"t":3087.525,"mix":1269},{"t":3088.771,"mix":1270},{"t":3090.342,"mix":1271},{"t":3091.983,"mix":1272},{"t":3093.196,"mix":1273},{"t":3094.656,"mix":1274},{"t":3095.8,"mix":1275},{"t":3096.732,"mix":1276},{"t":3098.153,"mix":1277},{"t":3099.371,"mix":1278},{"t":3100.236,"mix":1279},{"t":3101.042,"mix":1280},{"t":3101.824,"mix":1281},{"t":3102.59,"mix":1282},{"t":3110.821,"mix":1283},{"t":3114.723,"mix":1284},{"t":3120.261,"mix":1285},{"t":3124.684,"mix":1286},{"t":3129.539,"mix":1287},{"t":3134.375,"mix":1288},{"t":3138.041,"mix":1289},{"t":3142.295,"mix":1290},{"t":3148.171,"mix":1291},{"t":3160.062,"mix":1292},{"t":3164.806,"mix":1293},{"t":3169.167,"mix":1294},{"t":3173.216,"mix":1295},{"t":3177.047,"mix":1296},{"t":3181.043,"mix":1297},{"t":3186.237,"mix":1298},{"t":3191.433,"mix":1299},{"t":3195.354,"mix":1300},{"t":3199.331,"mix":1301},{"t":3203.881,"mix":1302},{"t":3208.511,"mix":1303},{"t":3212.992,"mix":1304},{"t":3216.711,"mix":1305},{"t":3221.305,"mix":1306},{"t":3225.103,"mix":1307},{"t":3228.788,"mix":1308},{"t":3232.849,"mix":1309},{"t":3236.559,"mix":1310},{"t":3240.213,"mix":1311},{"t":3243.66,"mix":1312},{"t":3248.502,"mix":1313},{"t":3252.005,"mix":1314},{"t":3256.759,"mix":1315},{"t":3260.561,"mix":1316},{"t":3265.274,"mix":1317},{"t":3268.689,"mix":1318},{"t":3271.396,"mix":1319},{"t":3277.671,"mix":1320},{"t":3281.293,"mix":1321},{"t":3283.892,"mix":1322},{"t":3287.351,"mix":1323},{"t":3290.166,"mix":1324},{"t":3293.485,"mix":1325},{"t":3296.159,"mix":1326},{"t":3299.029,"mix":1327},{"t":3302.526,"mix":1328},{"t":3306.03,"mix":1329},{"t":3308.767,"mix":1330},{"t":3312.927,"mix":1331},{"t":3315.952,"mix":1332},{"t":3319.426,"mix":1333},{"t":3322.127,"mix":1334},{"t":3324.938,"mix":1335},{"t":3327.66,"mix":1336},{"t":3331.157,"mix":1337},{"t":3333.78,"mix":1338},{"t":3337.701,"mix":1339},{"t":3347.541,"mix":1340},{"t":3352.878,"mix":1341},{"t":3357.12,"mix":1342},{"t":3361.165,"mix":1343},{"t":3366.197,"mix":1344},{"t":3370.107,"mix":1345},{"t":3373.808,"mix":1346},{"t":3377.492,"mix":1347},{"t":3381.295,"mix":1348},{"t":3385.695,"mix":1349},{"t":3388.527,"mix":1350},{"t":3391.695,"mix":1351},{"t":3394.849,"mix":1352},{"t":3398.275,"mix":1353},{"t":3401.241,"mix":1354},{"t":3404.256,"mix":1355},{"t":3408.184,"mix":1356},{"t":3411.652,"mix":1357},{"t":3413.885,"mix":1358},{"t":3416.333,"mix":1359},{"t":3421.596,"mix":1360},{"t":3424.373,"mix":1361},{"t":3429.632,"mix":1362},{"t":3431.884,"mix":1363},{"t":3433.96,"mix":1364},{"t":3435.478,"mix":1365},{"t":3437.461,"mix":1366},{"t":3438.942,"mix":1367},{"t":3440.949,"mix":1368},{"t":3442.57,"mix":1369},{"t":3444.512,"mix":1370},{"t":3446.243,"mix":1371},{"t":3448.273,"mix":1372},{"t":3449.805,"mix":1373},{"t":3451.502,"mix":1374},{"t":3452.856,"mix":1375},{"t":3454.523,"mix":1376},{"t":3455.815,"mix":1377},{"t":3457.519,"mix":1378},{"t":3459.174,"mix":1379},{"t":3462.009,"mix":1380},{"t":3465.29,"mix":1381},{"t":3467.707,"mix":1382},{"t":3470.864,"mix":1383},{"t":3473.219,"mix":1384},{"t":3475.343,"mix":1385},{"t":3477.62,"mix":1386},{"t":3479.729,"mix":1387},{"t":3481.664,"mix":1388},{"t":3483.649,"mix":1389},{"t":3485.365,"mix":1390},{"t":3487.179,"mix":1391},{"t":3489.981,"mix":1392},{"t":3492.128,"mix":1393},{"t":3494.292,"mix":1394},{"t":3496.091,"mix":1395},{"t":3499.028,"mix":1396},{"t":3501.469,"mix":1397},{"t":3503.718,"mix":1398},{"t":3506.144,"mix":1399},{"t":3508.321,"mix":1400},{"t":3511.312,"mix":1401},{"t":3514.38,"mix":1402},{"t":3516.393,"mix":1403},{"t":3518.448,"mix":1404},{"t":3520.249,"mix":1405},{"t":3521.688,"mix":1406},{"t":3523.704,"mix":1407},{"t":3525.768,"mix":1408},{"t":3527.662,"mix":1409},{"t":3529.746,"mix":1410},{"t":3531.693,"mix":1411},{"t":3534.201,"mix":1412},{"t":3536.825,"mix":1413},{"t":3539.985,"mix":1414},{"t":3542.578,"mix":1415},{"t":3545.38,"mix":1416},{"t":3549.501,"mix":1417},{"t":3553.651,"mix":1418},{"t":3556.364,"mix":1419},{"t":3559.507,"mix":1420},{"t":3562.278,"mix":1421},{"t":3565.334,"mix":1422},{"t":3568.019,"mix":1423},{"t":3571.664,"mix":1424},{"t":3575.272,"mix":1425},{"t":3581.851,"mix":1426},{"t":3585.351,"mix":1427},{"t":3588.117,"mix":1428},{"t":3593.172,"mix":1429},{"t":3597.437,"mix":1430},{"t":3603.788,"mix":1431},{"t":3609.347,"mix":1432},{"t":3613.486,"mix":1433},{"t":3618.726,"mix":1434},{"t":3626.304,"mix":1435},{"t":3634.563,"mix":1436},{"t":3644.493,"mix":1437},{"t":3658.991,"mix":1438},{"t":3663.065,"mix":1439},{"t":3667.373,"mix":1440},{"t":3671.864,"mix":1441},{"t":3675.84,"mix":1442},{"t":3679.664,"mix":1443},{"t":3681.618,"mix":1444},{"t":3683.361,"mix":1445},{"t":3687.524,"mix":1446},{"t":3691.944,"mix":1447},{"t":3694.299,"mix":1448},{"t":3696.135,"mix":1449},{"t":3700.307,"mix":1450},{"t":3704.191,"mix":1451},{"t":3706.489,"mix":1452},{"t":3708.643,"mix":1453},{"t":3712.124,"mix":1454},{"t":3715.721,"mix":1455},{"t":3719.232,"mix":1456},{"t":3720.853,"mix":1457},{"t":3722.715,"mix":1458},{"t":3726.156,"mix":1459},{"t":3730.523,"mix":1460},{"t":3733.677,"mix":1461},{"t":3737.218,"mix":1462},{"t":3740.565,"mix":1463},{"t":3744.042,"mix":1464},{"t":3745.963,"mix":1465},{"t":3747.555,"mix":1466},{"t":3751.281,"mix":1467},{"t":3755.182,"mix":1468},{"t":3758.608,"mix":1469},{"t":3761.832,"mix":1470},{"t":3765.102,"mix":1471},{"t":3768.492,"mix":1472},{"t":3770.021,"mix":1473},{"t":3771.968,"mix":1474},{"t":3774.553,"mix":1475},{"t":3777.074,"mix":1476},{"t":3778.432,"mix":1477},{"t":3779.677,"mix":1478},{"t":3781.945,"mix":1479},{"t":3783.989,"mix":1480},{"t":3787.774,"mix":1481},{"t":3791.346,"mix":1482},{"t":3792.705,"mix":1483},{"t":3794.289,"mix":1484},{"t":3797.555,"mix":1485},{"t":3800.537,"mix":1486},{"t":3804.113,"mix":1487},{"t":3805.568,"mix":1488},{"t":3807.631,"mix":1489},{"t":3811.246,"mix":1490},{"t":3815.015,"mix":1491},{"t":3819.459,"mix":1492},{"t":3824.532,"mix":1493},{"t":3829.288,"mix":1494},{"t":3835.494,"mix":1495},{"t":3838.837,"mix":1496},{"t":3841.985,"mix":1497},{"t":3844.615,"mix":1498},{"t":3848.375,"mix":1499},{"t":3852.103,"mix":1500},{"t":3855.174,"mix":1501},{"t":3858.247,"mix":1502},{"t":3861.329,"mix":1503},{"t":3864.488,"mix":1504},{"t":3867.484,"mix":1505},{"t":3870.414,"mix":1506},{"t":3873.668,"mix":1507},{"t":3876.6,"mix":1508},{"t":3879.023,"mix":1509},{"t":3881.489,"mix":1510},{"t":3883.639,"mix":1511},{"t":3886.886,"mix":1512},{"t":3890.179,"mix":1513},{"t":3892.809,"mix":1514},{"t":3895.473,"mix":1515},{"t":3903.529,"mix":1516},{"t":3906.982,"mix":1517},{"t":3909.791,"mix":1518},{"t":3912.959,"mix":1519},{"t":3916.631,"mix":1520},{"t":3919.935,"mix":1521},{"t":3923.278,"mix":1522},{"t":3926.618,"mix":1523},{"t":3930.399,"mix":1524},{"t":3934.14,"mix":1525},{"t":3936.956,"mix":1526},{"t":3940.314,"mix":1527},{"t":3943.686,"mix":1528},{"t":3949.705,"mix":1529},{"t":3954.038,"mix":1530},{"t":3986.191,"mix":1531}]
'''  # Use the full JSON
timestamps1 = json.loads(timestamps_json)

def parse_start_time_from_url(youtube_url):
    parsed_url = urlparse.urlparse(youtube_url)
    query_params = urlparse.parse_qs(parsed_url.query)
    start_time = query_params.get('t', ['0'])[0]  # Default to '0' if not provided
    if 's' in start_time:
        # Extract time in seconds from the URL parameter
        time_seconds = int(re.search(r'\d+', start_time).group())
        return time_seconds
    else:
        return int(start_time)

def download_audio_with_retries(youtube_url, output_path, max_retries=5):
    retry_count = 0
    while retry_count < max_retries:
        try:
            yt = YouTube(youtube_url)
            audio_stream = yt.streams.filter(only_audio=True).first()
            if not audio_stream:
                print("No audio stream found.")
                return None
            # Download the audio stream directly without conversion
            downloaded_file = audio_stream.download(filename=output_path)
            return downloaded_file
        except Exception as e:
            print(f"Attempt {retry_count + 1} failed: {str(e)}")
            time.sleep(5)  # wait 5 seconds before retrying
            retry_count += 1
    print("Failed to download after several retries.")
    return None

def convert_time_to_seconds(time):
    if isinstance(time, str) and ':' in time:
        minutes, seconds = map(int, time.split(':'))
        return minutes * 60 + seconds
    elif isinstance(time, (int, float)):
        return time
    else:
        raise ValueError("Time format must be a string 'MM:SS' or a number representing seconds")

def convert_to_ogg(input_path, output_path, start_time=0, end_time=None):
    try:
        # Convert start time to seconds and add offset
        start_total_seconds = convert_time_to_seconds(start_time) + 0.15
        
        # Initialize the ffmpeg command
        command = [
            'ffmpeg', '-ss', str(start_total_seconds), '-i', input_path,
            '-c:a', 'libvorbis', '-q:a', '5'
        ]
        
        if end_time is not None:
            # Convert end time to seconds
            end_total_seconds = convert_time_to_seconds(end_time)
            # Calculate the duration of the clip
            clip_duration = end_total_seconds - start_total_seconds
            command.extend(['-t', str(clip_duration)])
        
        command.append(output_path)
        
        subprocess.run(command, check=True)
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"Error during conversion: {e}")
        return None
    except ValueError as e:
        print(f"Invalid time format: {e}")
        return None

def load_audio_segment(audio_path, start_time=0, end_time=None, sr=11025):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
        duration = end_total_seconds - start_total_seconds
    else:
        # If end_time is None, calculate the full length of the audio
        full_duration = librosa.get_duration(filename=audio_path)
        duration = full_duration - start_total_seconds

    y, sr = librosa.load(audio_path, sr=sr, offset=start_total_seconds, duration=duration)
    return y, sr

# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=KsGLmrR0BVs'
youtube_url2 = 'https://www.youtube.com/watch?v=Ys7v789Lhoo'

start_time1 = 8.46
start_time2 = parse_start_time_from_url(youtube_url2)
end_time1 = "66:12"
end_time2 = None
full_duration1 = 0
full_duration2 = 0

# Download audio files
audio_path1 = download_audio_with_retries(youtube_url1, 'youtube_audio1.mp4')
if audio_path1:
    print('audio_path1 downloaded')
audio_path2 = download_audio_with_retries(youtube_url2, 'youtube_audio2.mp4')
if audio_path2:
    print('audio_path2 downloaded')


# Convert to OGG
ogg_path1 = "youtube_audio1.ogg"
ogg_path2 = "youtube_audio2.ogg"
#convert_to_ogg(audio_path1, ogg_path1, start_time1, end_time1)
convert_to_ogg(audio_path2, ogg_path2, start_time2, end_time2)

audio_path1 downloaded
audio_path2 downloaded


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

'youtube_audio2.ogg'

In [30]:
import gc
from concurrent.futures import ThreadPoolExecutor

HOP_LENGTH = 128
HOP_LENGTH_1 = 1024
HOP_LENGTH_2 = 256
OVERLAP_HOP = 256

def load_and_preprocess_audio_ORIGINAL(audio_path, target_sr=11025):
    # Load audio file at a reduced sample rate
    y, sr = librosa.load(audio_path, sr=target_sr)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256)
    log_S = librosa.power_to_db(S, ref=np.max)
    return log_S.T, sr

def load_and_preprocess_audio(audio_path, target_sr=11025, hop_length=HOP_LENGTH):
    # Load audio file at a higher sample rate for better temporal resolution
    y, sr = librosa.load(audio_path, sr=target_sr)
    
    # Harmonic-Percussive Source Separation (HPSS) --- cutting for resource savings
    #y_harmonic, y_percussive = librosa.effects.hpss(y)
    
    # Mel-spectrogram for harmonic component
    #S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=64, hop_length=HOP_LENGTH)
    #log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
    
    # Constant-Q Transform for better frequency resolution
    #CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    # Combine features
    #combined_features = np.vstack((log_S_harmonic, CQT))
    #print(f"Combined features shape: {combined_features.shape}")
    
    #return combined_features.T, sr

    #BELOW IS STRIPPED VERSION OF ABOVE
    y, sr = librosa.load(audio_path, sr=sr)
    
    # Mel-spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=512, hop_length=HOP_LENGTH)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Constant-Q Transform
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    combined_features = np.vstack((log_S, CQT))
    return combined_features.T, sr


def load_and_preprocess_audio_INCREMENTAL(audio_path, target_sr=11025, chunk_duration=10):
    # Load the audio file in chunks
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Harmonic-Percussive Source Separation (HPSS)
        y_harmonic, y_percussive = librosa.effects.hpss(y_chunk)
        
        # Mel-spectrogram for harmonic component
        S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=256)
        log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
        
        # Constant-Q Transform for better frequency resolution
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr)), ref=np.max)
        
        # Combine features
        combined_chunk_features = np.vstack((log_S_harmonic, CQT))
        
        combined_features.append(combined_chunk_features.T)
        
        # Print progress
        if (i + 1) % progress_step == 0:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    
    return combined_features, sr

def load_and_preprocess_audio_OVERLAP(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows
        chunk_features = []
        for offset in range(0, HOP_LENGTH, OVERLAP_HOP):
            if offset > 0:
                y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
            else:
                y_shifted = y_chunk
            
            # HPSS
            y_harmonic, y_percussive = librosa.effects.hpss(y_shifted)
            
            # Mel-spectrogram
            S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
            log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
            
            # Constant-Q Transform
            CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_shifted, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
            
            # Combine features for this offset
            combined_chunk_features = np.vstack((log_S_harmonic, CQT))
            chunk_features.append(combined_chunk_features.T)
            del y_shifted, y_harmonic, y_percussive, S_harmonic, log_S_harmonic, CQT
            gc.collect()

        combined_chunk_features = np.concatenate(chunk_features, axis=1)
        combined_features.append(combined_chunk_features)
        del chunk_features
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

#low and high averageed hoplengths
def load_and_preprocess_audio_AVERAGE(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH_2 // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows for both hop lengths
        chunk_features_list = []
        max_length = 0
        for hop_length in [HOP_LENGTH_1, HOP_LENGTH_2]:
            chunk_features = []
            for offset in range(0, hop_length, OVERLAP_HOP):
                if offset > 0:
                    y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
                else:
                    y_shifted = y_chunk
                
                # Mel-spectrogram
                S = librosa.feature.melspectrogram(y=y_shifted, sr=sr, n_mels=n_mels, hop_length=hop_length)
                log_S = librosa.power_to_db(S, ref=np.max)
                
                chunk_features.append(log_S.T)
                del y_shifted, S, log_S
                gc.collect()

            # Find the maximum length of the feature matrices
            max_length = max(max_length, max(f.shape[0] for f in chunk_features))
            chunk_features_list.append(chunk_features)
            del chunk_features
            gc.collect()
        
        # Resize all feature matrices to the maximum length and average
        resized_chunk_features_list = []
        for features in chunk_features_list:
            resized_features = [resize(f, (max_length, f.shape[1]), anti_aliasing=True) for f in features]
            averaged_chunk_features = np.mean(resized_features, axis=0)
            resized_chunk_features_list.append(averaged_chunk_features)
        
        # Average the features from different hop lengths
        combined_chunk_features = np.mean(resized_chunk_features_list, axis=0)
        combined_features.append(combined_chunk_features)
        del chunk_features_list, resized_chunk_features_list
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

def load_and_preprocess_audio_INCREMENTAL_NOHPSS(audio_path, target_sr=11025, chunk_duration=10, n_mels=256, mel_weight=1.0, cqt_weight=1.0, tempo_weight=1.0):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Mel-spectrogram
        S = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
        log_S = librosa.power_to_db(S, ref=np.max)
        
        # Constant-Q Transform
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

        # Tempogram
        onset_env = librosa.onset.onset_strength(y=y_chunk, sr=sr, hop_length=HOP_LENGTH)
        tempogram = librosa.feature.tempogram(onset_envelope=onset_env, sr=sr, hop_length=HOP_LENGTH)

        # Ensure the same length for both arrays
        min_length = min(log_S.shape[1], CQT.shape[1])
        log_S = log_S[:, :min_length]
        CQT = CQT[:, :min_length]
        
        # Normalize features
        log_S = normalize_features(log_S)
        CQT = normalize_features(CQT)

        log_S *= mel_weight
        CQT *= cqt_weight
        tempogram *= tempo_weight
        
        # Combine features
        combined_chunk_features = np.vstack((log_S, CQT))

        combined_features.append(combined_chunk_features.T)
        del log_S, CQT, y_chunk, S, combined_chunk_features
        gc.collect()
        
        # Print progress
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

def process_chunk(y_chunk, sr, hop_length):
    # Chroma feature extraction
    chroma = librosa.feature.chroma_stft(y=y_chunk, sr=sr, hop_length=HOP_LENGTH)
    # chroma = normalize_features(chroma)
    
    # Constant-Q Transform feature extraction
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
    # CQT = normalize_features(CQT)
    
    # Ensure the same length for both arrays
    min_length = min(chroma.shape[1], CQT.shape[1])
    chroma = chroma[:, :min_length]
    CQT = CQT[:, :min_length]
    
    # Combine features
    combined_chunk_features = np.vstack((chroma, CQT)).T
    
    return combined_chunk_features

def load_and_preprocess_audio_combined(audio_path, target_sr=44100, chunk_duration=10, hop_length=HOP_LENGTH):
    y, sr = librosa.load(audio_path, sr=target_sr)
    y = normalize_audio(y)  # Normalize the raw audio signal
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = max(1, num_chunks // 10)  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")

    def process_and_collect(i):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        chunk_features = process_chunk(y_chunk, sr, HOP_LENGTH)
        return chunk_features

    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_and_collect, i) for i in range(num_chunks)]
        for i, future in enumerate(futures):
            combined_features.append(future.result())
            # Print progress
            if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
                print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")
            gc.collect()

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

# Normalize the raw audio signal
def normalize_audio(y, epsilon=1e-8):
    return (y - np.mean(y)) / (np.std(y) + epsilon)
    
def normalize_features(features, epsilon=1e-8):
    mean = np.mean(features, axis=0)
    std_dev = np.std(features, axis=0)
    return (features - mean) / (std_dev + epsilon)

    
# Load and preprocess both audio recordings in OGG format

S1, sr1 = load_and_preprocess_audio_combined(ogg_path1)
#S1, sr1 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path1, mel_weight=3.0, cqt_weight=1.0, tempo_weight=2.0)

Total chunks: 397, Progress step: 39
Processed 10% of chunks
Processed 20% of chunks
Processed 29% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 88% of chunks
Processed 98% of chunks
Processed 100% of chunks
Combined features shape: (1365784, 96)


In [22]:
#making own step so no need to re-run audio1 for multiple recordings.

S2, sr2 = load_and_preprocess_audio_combined(ogg_path2)
#S2, sr2 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path2, mel_weight=3.0, cqt_weight=1.0, tempo_weight=2.0)

Total chunks: 377, Progress step: 37
Processed 10% of chunks
Processed 20% of chunks
Processed 29% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 88% of chunks
Processed 98% of chunks
Processed 100% of chunks
Combined features shape: (1298879, 96)


In [23]:
print(S1.shape,S2.shape)

(1365784, 96) (1298879, 96)


In [24]:
from fastdtw import fastdtw
#from dtaidistance import dtw
    
def dynamic_time_warping_approx(S1, S2):
    distance, path = fastdtw(S1, S2)
    return path

#def constrained_dtw(S1, S2):
#    # Calculate the DTW distance with a window constraint
#    distance, paths = dtw.warping_paths(S1, S2, window=1000)
#    # Find the best path
#    path = dtw.best_path(paths)
#    return path

#warping_path = constrained_dtw(S1, S2)
warping_path = dynamic_time_warping_approx(S1, S2)

In [25]:
def adjust_timestamps(wp, timestamps, sr):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    
    for entry in timestamps:
        original_frame = int((entry['t']) * sr / HOP_LENGTH)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * HOP_LENGTH / sr
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    
    # Ensure the last timestamp is included and set "t" to 9999
    if timestamps:
        last_entry = timestamps[-1]
        last_entry_adjusted = {"t": 9999, "mix": last_entry['mix']}
        if adjusted_timestamps and adjusted_timestamps[-1]['mix'] == last_entry['mix']:
            adjusted_timestamps[-1] = last_entry_adjusted
        else:
            adjusted_timestamps.append(last_entry_adjusted)
    
    return adjusted_timestamps

# Sample JSON timestamps for the first recording (assumed already loaded)
adjusted_timestamps = adjust_timestamps(warping_path, timestamps1, sr1)

In [26]:
def calculate_ratios(timestamps):
    ratios = []
    for i in range(1, len(timestamps)):
        current_ratio = abs(timestamps[i]['t'] - timestamps[i-1]['t']) if timestamps[i-1]['t'] != 0 else 0
        ratios.append(current_ratio)
    return ratios

def compare_and_flag_changes(adjusted_timestamps, original_timestamps, audio_length, neighbor_count=5):
    # Set last timestamp as per new requirement
    adjusted_timestamps[-1]['t'] = audio_length + 1

    # Calculate differences and ratios
    adjusted_ratios = calculate_ratios(adjusted_timestamps)
    original_ratios = calculate_ratios(original_timestamps)

    # Array to hold timestamps that are significantly different
    flagged_timestamps = []

    # Analyze ratios for significant changes
    for i in range(len(adjusted_ratios) - 1):  # Ignore the last timestamp in comparison
        start = max(0, i - neighbor_count)
        end = min(len(original_ratios) - 1, i + neighbor_count + 1)  # Avoid including the last in comparison
        
        # Calculate neighborhood average without including out-of-range values
        neighborhood_original = original_ratios[start:end]
        if not neighborhood_original:
            continue
        neighborhood_average = np.mean(neighborhood_original)
        
        # Check if the current adjusted ratio is significantly different
        if adjusted_ratios[i] > 1.5 * neighborhood_average:
            flagged_timestamps.append({
                "mix": adjusted_timestamps[i]['mix'],
                "original_ratio": original_ratios[i] if i < len(original_ratios) else 0,
                "adjusted_ratio": adjusted_ratios[i],
                "average_neighbors": neighborhood_average
            })

    return flagged_timestamps

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']

# Initialize an empty list to store the new adjusted timestamps
new_adjusted_timestamps = []
previous_t = None  # Variable to hold the previous timestamp

for item in adjusted_timestamps:
    adjusted_t = round(item["t"] + initial_offset, 3)
    new_adjusted_timestamps.append({"t": adjusted_t, "mix": item["mix"]})
    
    # Check if the previous timestamp is defined and compare the current timestamp with the previous one
    if previous_t is not None and (adjusted_t - previous_t < 0.5):
        difference = adjusted_t - previous_t
        print(f"Close timestamps found: Mix: {item['mix']}, Difference: {difference:.3f}, Previous - {previous_t}, Current - {adjusted_t}")
    
    # Update the previous_t to the current timestamp for the next iteration
    previous_t = adjusted_t


# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps, indent=4))
# Here we adjust to show the total offset from the original video start
full_offset = abs(initial_offset) + abs(start_time2)
print(f"Total Offset from Video Start: {full_offset}, initial {initial_offset} + start_time2 {start_time2}")

def calculate_full_duration(audio_path, start_time, end_time):
    start_total_seconds = convert_time_to_seconds(start_time)
    
    if end_time is not None:
        end_total_seconds = convert_time_to_seconds(end_time)
    else:
        # Calculate the full length of the audio if end_time is None
        full_duration = librosa.get_duration(path=audio_path)
        end_total_seconds = full_duration

    full_duration = end_total_seconds - start_total_seconds
    return full_duration

full_duration2 = calculate_full_duration(ogg_path2, start_time2, end_time2)

#Flag anything that exceeds 50% difference compared to neighboring measures.
flagged_timestamps = compare_and_flag_changes(new_adjusted_timestamps, timestamps1, full_duration2)
print("Flagged Timestamps:")
for ft in flagged_timestamps:
    print(f"Mix: {ft['mix']}, Original Ratio: {ft['original_ratio']:.3f}, Adjusted Ratio: {ft['adjusted_ratio']:.3f}, Neighbors' Avg.: {ft['average_neighbors']:.3f}")

# Cleanup downloaded and converted files
#os.remove(audio_path1)
#os.remove(audio_path2)
#os.remove(ogg_path1)
#os.remove(ogg_path2)

Close timestamps found: Mix: 93, Difference: 0.122, Previous - 128.467, Current - 128.589
Close timestamps found: Mix: 94, Difference: 0.000, Previous - 128.589, Current - 128.589
Close timestamps found: Mix: 95, Difference: 0.009, Previous - 128.589, Current - 128.598
Close timestamps found: Mix: 96, Difference: 0.366, Previous - 128.598, Current - 128.964
Close timestamps found: Mix: 115, Difference: 0.270, Previous - 150.454, Current - 150.724
Close timestamps found: Mix: 331, Difference: 0.479, Previous - 724.75, Current - 725.229
Close timestamps found: Mix: 414, Difference: 0.000, Previous - 876.449, Current - 876.449
Close timestamps found: Mix: 427, Difference: 0.391, Previous - 919.351, Current - 919.742
Close timestamps found: Mix: 482, Difference: 0.006, Previous - 1018.154, Current - 1018.16
Close timestamps found: Mix: 502, Difference: 0.471, Previous - 1046.311, Current - 1046.782
Close timestamps found: Mix: 539, Difference: 0.345, Previous - 1115.696, Current - 1116.041